# EasyMagpieTTS — offline two-stage synthesis

This notebook creates **one** `AsyncOmni` engine and reuses it for three self-contained API examples:

1. synthesis with a saved speaker embedding;
2. synthesis with request-time reference audio;
3. a two-turn dialogue with raw user-audio history.

Each inference cell defines its own inputs, builds the exact prompt passed to `omni.generate(...)`, invokes the engine, and saves its output. Run the cells from top to bottom; the final cell shuts the engine down. First [convert the NeMo checkpoint](../../../tools/easymagpie_vllm_omni/README.md#convert-a-nemo-checkpoint) with `--bundle-audio-encoders` for raw-audio input and [install the serving environment](../../../tools/easymagpie_vllm_omni/README.md#setup-the-serving-environment).


In [ ]:
import asyncio
import json
import os
import uuid
from collections import Counter
from pathlib import Path

os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import vllm_plugin_easymagpie_omni
vllm_plugin_easymagpie_omni.register()

import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer
from vllm import SamplingParams
from vllm.engine.protocol import StreamingInput
from vllm.sampling_params import RequestOutputKind
from vllm_omni import AsyncOmni

from easymagpie_vllm_omni.audio_output import extract_audio_from_stage_output, is_audio_segment_finished
from easymagpie_vllm_omni.config import EasyMagpieOmniArch
from easymagpie_vllm_omni.easymagpie import EasyMagpieTTSForConditionalGeneration


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "easymagpie_vllm_omni").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within a Speech repository checkout")


REPO_ROOT = find_repo_root()
EASYMAGPIE_ROOT = REPO_ROOT / "tools" / "easymagpie_vllm_omni"
MODEL_DIR = str(EASYMAGPIE_ROOT / "converted_model")
DEPLOY_CONFIG = str(EASYMAGPIE_ROOT / "deploy" / "easymagpie.yaml")
MAX_NEW_TOKENS = 1024


## Shared model and output helpers

This cell reads the converted-model contract and defines sampling/audio helpers. Raw audio is intentionally not downmixed or resampled: every audio cell passes an explicit `(mono_waveform, sample_rate)` tuple, and the rate must match `codec_input_sample_rate` in the converted config.


In [ ]:
model_config = json.loads((Path(MODEL_DIR) / "config.json").read_text())
model_config_type = type("EasyMagpieConfig", (), model_config)
arch = EasyMagpieOmniArch.from_hf_config(model_config_type)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
context_ids = tokenizer.encode("[EN]")
task_rows = int(arch.num_task_embeddings > 0)
stop_token_id = EasyMagpieTTSForConditionalGeneration.audio_eos_stop_token_id(model_config_type)

lm_sampling = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    detokenize=False,
    ignore_eos=False,
    stop_token_ids=[stop_token_id],
    output_kind=RequestOutputKind.DELTA,
)
codec_sampling = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    detokenize=True,
    output_kind=RequestOutputKind.DELTA,
)


def read_codec_audio(path: Path):
    waveform, sample_rate = sf.read(path, dtype="float32")
    if waveform.ndim != 1:
        raise ValueError(f"{path} must be mono; received shape {waveform.shape}")
    if sample_rate != arch.codec_input_sample_rate:
        raise ValueError(
            f"{path} must be {arch.codec_input_sample_rate} Hz for this codec; received {sample_rate} Hz"
        )
    return waveform, sample_rate


def save_and_show(path: str, audio):
    waveform, sample_rate = audio
    sf.write(path, waveform, sample_rate)
    print(f"Wrote {path} ({len(waveform) / sample_rate:.2f}s @ {sample_rate} Hz)")
    display(Audio(waveform, rate=sample_rate))


async def collect_audio(stage_outputs, request_id: str):
    stage_counts = Counter()
    chunks = []
    sample_rate = None
    try:
        async for stage_output in stage_outputs:
            stage_counts[getattr(stage_output, "stage_id", "unknown")] += 1
            extracted = extract_audio_from_stage_output(stage_output)
            if extracted is not None:
                chunk, chunk_rate = extracted
                if sample_rate is not None and chunk_rate != sample_rate:
                    raise RuntimeError(f"decoded sample rate changed from {sample_rate} to {chunk_rate}")
                sample_rate = chunk_rate
                chunks.append(np.asarray(chunk))
    except Exception as error:
        raise RuntimeError(f"{request_id} failed after stage outputs {dict(stage_counts)}") from error
    if not chunks:
        raise RuntimeError(f"{request_id} completed without decoded audio; stage outputs={dict(stage_counts)}")
    return np.concatenate(chunks), sample_rate


## Start one reusable engine

This is the only expensive initialization in the notebook. All following request cells reuse `omni`. If you need to recreate it, run the shutdown cell at the bottom first.


In [ ]:
if globals().get("omni") is not None:
    raise RuntimeError("An engine already exists. Run the shutdown cell before recreating it.")

omni = AsyncOmni(model=MODEL_DIR, deploy_config=DEPLOY_CONFIG, log_stats=False)
print(f"Engine ready: {MODEL_DIR}")


## 1. Saved speaker embedding

This cell owns the saved-speaker inputs. It selects `speaker_embeddings/<SPEAKER_ID>.pt` from the converted model, composes the text-only prompt, and invokes the shared engine.


In [ ]:
SPEAKER_ID = "eng"
TEXT = "Hello, welcome to the text-to-speech demo."
OUT_WAV = "out.wav"

prompt_len = EasyMagpieTTSForConditionalGeneration.get_prompt_len(
    SPEAKER_ID, MODEL_DIR, tokenize=lambda value: tokenizer.encode(value)
)
prompt = {
    "prompt_token_ids": [0] * prompt_len,
    "additional_information": {
        "context_text": "[EN]",
        "text": TEXT,
        "temperature": 0.7,
        "top_k": 80,
        "speaker_id": SPEAKER_ID,
    },
}

request_id = f"easymp-saved-speaker-{uuid.uuid4().hex[:8]}"
stage_outputs = omni.generate(
    prompt,
    sampling_params_list=[lm_sampling, codec_sampling],
    request_id=request_id,
)
audio = await collect_audio(stage_outputs, request_id)
save_and_show(OUT_WAV, audio)


## 2. Request-time reference audio

This cell owns the zero-shot inputs. The AN4 waveform supplies only the output voice; its transcript is documented for listening but is not sent to the model. The converted artifact must have been created with `--bundle-audio-encoders`, which packages both the codec encoder and the reference-speaker Transformer.


In [ ]:
REFERENCE_AUDIO = REPO_ROOT / "tests/.data/an4_speaker/an4/wav/an4_clstk/fash/cen5-fash-b.wav"
# Documentation only; this transcript is not part of the request.
REFERENCE_AUDIO_TRANSCRIPT = "P I T T S B U R G H"
TEXT = "This voice is conditioned from request-time reference audio."
REFERENCE_OUT_WAV = "out_reference_audio.wav"

arch.require_reference_audio(MODEL_DIR)
if not REFERENCE_AUDIO.is_file():
    raise FileNotFoundError(f"Missing repository test audio: {REFERENCE_AUDIO}")

reference_audio = read_codec_audio(REFERENCE_AUDIO)
target_ids = tokenizer.encode(TEXT)
if len(target_ids) < arch.text_prefill_num:
    raise ValueError("Text is too short for this checkpoint's causal text prefill")

prompt = {
    "prompt_token_ids": (
        [0] * task_rows
        + [arch.audio_input_token_id]
        + [0] * len(context_ids)
        + [0] * arch.text_prefill_num
    ),
    "multi_modal_data": {"audio": [reference_audio]},
    "mm_processor_kwargs": {"audio_roles": ["speaker_reference"]},
    "additional_information": {
        "context_text": "[EN]",
        "speaker_reference_audio": True,
        "text": TEXT,
        "text_prefill_num": arch.text_prefill_num,
        "prefill_text_tokens": target_ids[: arch.text_prefill_num],
        "temperature": 0.7,
        "top_k": 80,
    },
}

request_id = f"easymp-reference-audio-{uuid.uuid4().hex[:8]}"
stage_outputs = omni.generate(
    prompt,
    sampling_params_list=[lm_sampling, codec_sampling],
    request_id=request_id,
)
audio = await collect_audio(stage_outputs, request_id)
save_and_show(REFERENCE_OUT_WAV, audio)


## 3. Two-turn dialogue with raw user-audio history

This is one resumable `omni.generate(...)` request containing two `StreamingInput` items.

- The **assistant voice** is cloned once from an FFMM AN4 recording that spells “P I T T S B U R G H.”
- **User turn 1** says “yes”; the model generates assistant reply 1.
- **User turn 2** says “go”; the same Stage-0 request retains the first user/assistant history and generates assistant reply 2.
- Stage 1 resets its response-local codec state at each dialogue boundary; it never needs the Stage-0 conversation history.

The user and reference clips are different speakers. Their transcripts are comments only and are not sent. The capability check first verifies the source checkpoint’s independent multi-turn flags, then verifies that conversion bundled the raw-audio tower. A bundled single-turn artifact still supports the previous zero-shot cell but will stop here with a clear error.


In [ ]:
SPEAKER_REFERENCE_AUDIO = REPO_ROOT / "tests/.data/an4_speaker/an4/wav/an4_clstk/ffmm/cen5-ffmm-b.wav"
SPEAKER_REFERENCE_TRANSCRIPT = "P I T T S B U R G H"  # Documentation only.

USER_TURN_1_AUDIO = REPO_ROOT / "tests/.data/an4_speaker/an4/wav/an4_clstk/fash/an251-fash-b.wav"
USER_TURN_1_TRANSCRIPT = "Yes."  # Documentation only.
ASSISTANT_TURN_1_TEXT = "Great. Shall I begin now?"

USER_TURN_2_AUDIO = REPO_ROOT / "tests/.data/an4_speaker/an4/wav/an4_clstk/fash/an253-fash-b.wav"
USER_TURN_2_TRANSCRIPT = "Go."  # Documentation only.
ASSISTANT_TURN_2_TEXT = "All right. I will begin now."

MULTITURN_OUT_WAVS = ["out_multiturn_turn1.wav", "out_multiturn_turn2.wav"]

arch.require_user_audio_prefill(MODEL_DIR)
for audio_path in (SPEAKER_REFERENCE_AUDIO, USER_TURN_1_AUDIO, USER_TURN_2_AUDIO):
    if not audio_path.is_file():
        raise FileNotFoundError(f"Missing repository test audio: {audio_path}")

speaker_reference = read_codec_audio(SPEAKER_REFERENCE_AUDIO)
user_turn_audio = [
    read_codec_audio(USER_TURN_1_AUDIO),
    read_codec_audio(USER_TURN_2_AUDIO),
]

turn_1_prompt = {
    "prompt_token_ids": (
        [0] * task_rows
        + [arch.audio_input_token_id]
        + [0] * len(context_ids)
        + [arch.audio_input_token_id]
    ),
    "multi_modal_data": {"audio": [speaker_reference, user_turn_audio[0]]},
    "mm_processor_kwargs": {"audio_roles": ["speaker_reference", "user"]},
    "additional_information": {
        "context_text": "[EN]",
        "speaker_reference_audio": True,
        "user_audio_prefill": True,
        "text": ASSISTANT_TURN_1_TEXT,
        "text_prefill_num": arch.text_prefill_num,
        "temperature": 0.7,
        "top_k": 80,
        "reset_codec_on_segment": True,
    },
}
turn_2_prompt = {
    # Stage 0 appends this user-audio marker to the existing dialogue state.
    "prompt_token_ids": [arch.audio_input_token_id],
    "multi_modal_data": {"audio": [user_turn_audio[1]]},
    "mm_processor_kwargs": {"audio_roles": ["user"]},
    "additional_information": {
        "speaker_reference_audio": False,
        "user_audio_prefill": True,
        "text": ASSISTANT_TURN_2_TEXT,
        "text_prefill_num": arch.text_prefill_num,
        "temperature": 0.7,
        "top_k": 80,
        "reset_codec_on_segment": True,
    },
}


turn_finished = [asyncio.Event(), asyncio.Event()]


async def dialogue_inputs():
    yield StreamingInput(prompt=turn_1_prompt, sampling_params=lm_sampling)
    # Submit user turn 2 only after assistant turn 1 has been fully decoded.
    await turn_finished[0].wait()
    yield StreamingInput(prompt=turn_2_prompt, sampling_params=lm_sampling)
    # Keep the stream open until assistant turn 2 has been fully decoded.
    await turn_finished[1].wait()


async def collect_dialogue(stage_outputs, request_id: str):
    stage_counts = Counter()
    turn_chunks = []
    current_chunks = []
    sample_rate = None

    try:
        async for stage_output in stage_outputs:
            stage_id = getattr(stage_output, "stage_id", "unknown")
            stage_counts[stage_id] += 1

            extracted = extract_audio_from_stage_output(stage_output)
            if extracted is not None:
                chunk, chunk_rate = extracted
                if sample_rate not in (None, chunk_rate):
                    raise RuntimeError("decoded sample rate changed within the dialogue")
                sample_rate = chunk_rate
                current_chunks.append(np.asarray(chunk))

            if stage_id == 1 and is_audio_segment_finished(stage_output):
                if not current_chunks:
                    raise RuntimeError("a dialogue turn finished without decoded audio")
                turn_index = len(turn_chunks)
                if turn_index >= len(turn_finished):
                    raise RuntimeError("the dialogue produced an unexpected extra decoded turn")
                turn_chunks.append(np.concatenate(current_chunks))
                current_chunks = []
                turn_finished[turn_index].set()
    except Exception as error:
        raise RuntimeError(f"{request_id} failed after stage outputs {dict(stage_counts)}") from error

    if len(turn_chunks) != 2:
        raise RuntimeError(
            f"{request_id} completed {len(turn_chunks)} decoded turns instead of 2; "
            f"stage outputs={dict(stage_counts)}"
        )
    return [(waveform, sample_rate) for waveform in turn_chunks]


request_id = f"easymp-multiturn-{uuid.uuid4().hex[:8]}"
stage_outputs = omni.generate(
    dialogue_inputs(),
    sampling_params_list=[lm_sampling, codec_sampling],
    request_id=request_id,
)
assistant_turn_audio = await collect_dialogue(stage_outputs, request_id)

dialogue = [
    ("User 1", USER_TURN_1_TRANSCRIPT, user_turn_audio[0], None),
    ("Assistant 1", ASSISTANT_TURN_1_TEXT, assistant_turn_audio[0], MULTITURN_OUT_WAVS[0]),
    ("User 2", USER_TURN_2_TRANSCRIPT, user_turn_audio[1], None),
    ("Assistant 2", ASSISTANT_TURN_2_TEXT, assistant_turn_audio[1], MULTITURN_OUT_WAVS[1]),
]
for role, transcript, turn_audio, output_path in dialogue:
    print(f"{role}: {transcript}")
    if output_path is None:
        display(Audio(turn_audio[0], rate=turn_audio[1]))
    else:
        save_and_show(output_path, turn_audio)


## Shut down

Run this after the requests you want to try. Restarting the kernel also stops the engine, but an explicit shutdown releases GPU resources cleanly.


In [ ]:
omni.shutdown()
omni = None
print("Engine shut down")
